In [1]:
import sys
print(sys.path)


import aqua
print(aqua.__file__)
from aqua.batchAQUA_general import batchAQUA
from aqua.utils import *
from aqua.stimulus import *


['/home/liam/anaconda3/envs/aqua/lib/python314.zip', '/home/liam/anaconda3/envs/aqua/lib/python3.14', '/home/liam/anaconda3/envs/aqua/lib/python3.14/lib-dynload', '', '/home/liam/anaconda3/envs/aqua/lib/python3.14/site-packages']
/home/liam/Documents/PhD_autapses/AQUA_project/AQUA_package/aqua/__init__.py


In [2]:
'''
Will test the STA protocol on filtered white noise injected into the RS_int and RS_res neurons

- need to find the threshold
- Apply a white noise stimulus to this threshold current.

'''

'\nWill test the STA protocol on filtered white noise injected into the RS_int and RS_res neurons\n\n- need to find the threshold\n- Apply a white noise stimulus to this threshold current.\n\n'

In [3]:
RS_int = {'name': 'RS_ref', 'C': 100, 'k': 0.7, 'v_r': -60, 'v_t': -40, 'v_peak': 35,
     'a': 0.03, 'b': -2, 'c': -50, 'd': 100, 'e': 0., 'f': 0., 'tau': 0.}

RS_res = {'name': 'RS_ref', 'C': 100, 'k': 0.7, 'v_r': -60, 'v_t': -40, 'v_peak': 35,
     'a': 0.03, 'b': 5, 'c': -50, 'd': 100, 'e': 0., 'f': 0., 'tau': 0.}

N_neurons = 50

T = 2000    # ms
dt = 0.1    # ms
N_iter = int(T/dt)

params_int = []
params_res = []

for i in range(N_neurons):
    
    params_int.append(RS_int)
    params_res.append(RS_res)


batch_int = batchAQUA(params_int)
batch_res = batchAQUA(params_res)

# find threshold
threshold_int, steady_int = batch_int.get_threshold(0)
threshold_res, steady_res = batch_res.get_threshold(0)

x_start_int = np.full((N_neurons, 3), fill_value = stead_int)
x_start_res = np.full((N_neurons, 3), fill_value = stead_res)
t_start = np.zeros(N_neurons)


# initialise
batch_int.Initialise(x_start_int, t_start)
batch_res.Initialise(x_start_res, t_start)

# create white noise input
I_wn = np.random.uniform((N_neurons, N_iter))
noise_scale = 20
I_int = threshold_int + noise_scale*I_wn
I_res = threshold_res + noise_scale*I_wn

# simulate each neuron with noise
X_int, T, spikes_int = batch_int.update_batch(dt, N_iter, I_int)
X_res, T, spikes_res = batch_res.update_batch(dt, N_iter, I_res)


window = 50
STA_int = STA(spikes_int, I_int, dt, window = window)
STA_res = STA(spikes_res, I_res, dt, window = window)



100%|██████████| 39999/39999 [00:05<00:00, 7789.78it/s]


IndexError: index -1 is out of bounds for axis 0 with size 0

In [ ]:
fig, ax = plt.subplots(2, 1, figsize = (10, 8))

time = np.linspace(0, dt*window, window)

for i in range(N_neurons):
    ax[0].plot(time, STA_int[i, :])
    ax[1].plot(time, STA_res[i, :])

ax[0].set_xlabel('Time [ms]')
ax[0].set_ylabel('STA')
ax[0].set_title('RS integrator')

ax[1].set_xlabel('Time [ms]')
ax[1].set_ylabel('STA')
ax[1].set_title('RS resonator')